In [ ]:
from aggregator import aggregate_events_metrics
from src.reader import read_events
from src.validator import validate_event, filter_event
from src.writer import write_stats_csv, write_summary_json, write_deadletter_json
from datetime import datetime, timezone, timedelta

from processor import process_events

In [2]:
execution_time = datetime.now(timezone.utc)
window_days = 30
input_dir = "events/"
output_path = 'resultados_teste/stats.csv'
deadletter_path = 'resultados_teste/deadletter.json'

In [3]:
events = read_events("events/")
print(f"Total de eventos lidos: {len(events)}")
events

Total de eventos lidos: 5


[{'user_id': 'user1',
  'event_type': 'login',
  'timestamp': '2025-09-01T10:00:00Z',
  'amount': None},
 {'user_id': 'user2',
  'event_type': 'purchase',
  'timestamp': '2025-09-01T12:00:00Z',
  'amount': 200.0},
 {'user_id': 'user3',
  'event_type': 'logout',
  'timestamp': '2025-07-01T16:00:00Z',
  'amount': None},
 {'user_id': '',
  'event_type': 'login',
  'timestamp': '2025-09-01T09:00:00Z',
  'amount': None},
 {'user_id': 'user4',
  'event_type': 'purchase',
  'timestamp': '2025-09-01T11:00:00Z',
  'amount': -50.0}]

In [4]:
valid_events = []
dead_letter_events = []

for event in events:
    is_valid, reason = validate_event(
        event=event,
        execution_time=execution_time
    )
    print(f"User ID: {event.get('user_id', 'N/A')} - Válido: {is_valid} - Motivo: {reason}")
    if is_valid:
        valid_events.append(event)
    else:
        dead_letter_events.append({"event": event, "reason": reason})


print(f"Total de eventos válidos: {len(valid_events)}")
print(f"Total de eventos no dead letter: {len(dead_letter_events)}")

User ID: user1 - Válido: True - Motivo: 
User ID: user2 - Válido: True - Motivo: 
User ID: user3 - Válido: True - Motivo: 
User ID:  - Válido: False - Motivo: Campo user_id não pode ser vazio.
User ID: user4 - Válido: False - Motivo: Campo amount é obrigatório para purchase e deve ser positivo.
User ID: user4 - Válido: True - Motivo: 
Total de eventos válidos: 4
Total de eventos no dead letter: 2


In [5]:
filtered_events = [
    event for event in valid_events
    if filter_event(
        event=event,
        execution_time=execution_time,
        window_days=window_days
    )
]
print(f"Total de eventos após filtro: {len(filtered_events)}")
filtered_events

Total de eventos após filtro: 1


[{'user_id': 'user4',
  'event_type': 'purchase',
  'timestamp': '2026-04-20T11:00:00Z',
  'amount': 1000.0}]

In [ ]:
stats = aggregate_events_metrics(events=filtered_events)
stats

{'unique_users': 1,
 'event_counts': {'login': 0, 'purchase': 1, 'logout': 0},
 'total_purchase_amount': 1000.0,
 'average_purchase_amount': 1000.0,
 'purchase_by_user': {'user4': 1000.0}}

In [7]:
start_window_date = execution_time.strftime("%Y-%m-%d")
final_window_date = (execution_time - timedelta(days=window_days)).strftime("%Y-%m-%d")

window = {
    "from": final_window_date,
    "to": start_window_date
}
print(f"Janela de análise: {window}")

Janela de análise: {'from': '2026-04-11', 'to': '2026-05-11'}


In [3]:
input_dir = "events/"
output_path = 'resultados_teste/stats.csv'
deadletter_path = 'resultados_teste/deadletter.json'
window_days = 30

In [9]:
write_stats_csv(
    stats=stats,
    output_path=output_path
)

In [10]:
write_summary_json(
    stats=stats,
    window=window,
    dead_letter_count=len(dead_letter_events),
    output_path=output_path.replace("stats.csv", "summary.json")
)

In [11]:
write_deadletter_json(
    dead_letter_events=dead_letter_events,
    output_path=deadletter_path
)

In [3]:
process_events(
    input_dir=input_dir,
    output_path=output_path,
    deadletter_path=deadletter_path,
    window_days=window_days
)

Processamento concluído.
Eventos lidos: 5
Eventos válidos: 3
Eventos filtrados: 0
Eventos no deadletter: 2
